### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and
associated metadata. It has two attributes:
page _ content: a string representing the content;
metadata: a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship
to other documents, and other information. Note that an individual Document object often
represents a chunk of a larger document.
Let's generate some sample documents:

In [27]:
from langchain_core.documents import Document

documents = [
    Document(page_content="This is the first document .knmbdhjiabedhbjad , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module", 
             metadata={"source": "doc2.txt"}),
    Document(page_content="This is the first document . jhldbjlhBDHJLBawd, it is something I dont know about I m just writing something random just to see whaty we will be doing the following module", 
             metadata={"source": "doc1.txt"}),
    Document(page_content="This is the first document .oyui qgwyuhbaedhbowseaf , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module", 
            metadata={"source": "doc3.txt"}),
]

In [28]:
!pip install langchain-chroma


In [29]:
documents

[Document(metadata={'source': 'doc2.txt'}, page_content='This is the first document .knmbdhjiabedhbjad , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module'),
 Document(metadata={'source': 'doc1.txt'}, page_content='This is the first document . jhldbjlhBDHJLBawd, it is something I dont know about I m just writing something random just to see whaty we will be doing the following module'),
 Document(metadata={'source': 'doc3.txt'}, page_content='This is the first document .oyui qgwyuhbaedhbowseaf , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module')]

In [30]:
import os 
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(model = "openai/gpt-oss-20b",groq_api_key = groq_api_key)

In [31]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5149.94it/s]


In [32]:
## Vector Store 
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embedding=embeddings)
vectorstore

In [33]:
vectorstore.similarity_search("This is the first document .knmbdhjiabedhbjad , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module", k=2)

[Document(id='cbd595d9-3f47-4797-b4e9-d514801d8bac', metadata={'source': 'doc2.txt'}, page_content='This is the first document .knmbdhjiabedhbjad , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module'),
 Document(id='10a5ffa3-898f-421b-b540-5de737c3ff0a', metadata={'source': 'doc2.txt'}, page_content='This is the first document .knmbdhjiabedhbjad , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module')]

In [34]:
## Async Query 

await vectorstore.asimilarity_search("This is the first document .knmbdhjiabedhbjad , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module", k=2)

[Document(id='cbd595d9-3f47-4797-b4e9-d514801d8bac', metadata={'source': 'doc2.txt'}, page_content='This is the first document .knmbdhjiabedhbjad , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module'),
 Document(id='10a5ffa3-898f-421b-b540-5de737c3ff0a', metadata={'source': 'doc2.txt'}, page_content='This is the first document .knmbdhjiabedhbjad , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module')]

In [35]:
vectorstore.similarity_search_with_score("This is the first document .knmbdhjiabedhbjad , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module", k=2)

[(Document(id='cbd595d9-3f47-4797-b4e9-d514801d8bac', metadata={'source': 'doc2.txt'}, page_content='This is the first document .knmbdhjiabedhbjad , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module'),
  2.432726871511026e-13),
 (Document(id='10a5ffa3-898f-421b-b540-5de737c3ff0a', metadata={'source': 'doc2.txt'}, page_content='This is the first document .knmbdhjiabedhbjad , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module'),
  2.432726871511026e-13)]

### Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into
LangChain Expression Language chains.
LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and
asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.
We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we
wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the
similarity_search method:

In [36]:
from typing import List 

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["knmbdhjiabedhbjad"])

[[Document(id='cbd595d9-3f47-4797-b4e9-d514801d8bac', metadata={'source': 'doc2.txt'}, page_content='This is the first document .knmbdhjiabedhbjad , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module')]]

Vectorstores implement an as_retrievec method that will generate a Retriever, specifically a
VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify
what methods of the underlying vector store to call, and how to parameterize them. For instance, we can
replicate the above with the following:

In [37]:
retriever =vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k": 1}
)

retriever.batch(["knmbdhjiabedhbjad"])

[[Document(id='cbd595d9-3f47-4797-b4e9-d514801d8bac', metadata={'source': 'doc2.txt'}, page_content='This is the first document .knmbdhjiabedhbjad , it is something I dont know about I m just writing something random just to see whaty we will be doing the following module')]]

In [39]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

res = rag_chain.invoke("knmbdhjiabedhbjad")
print(res.content)


The string **knmbdhjiabedhbjad** appears in the text of the document as a random, meaningless sequence of characters. It is not a word or a reference to anything else in the provided context.
